In [8]:
import os, re
import pathlib
from datetime import date, timedelta

import sys
import duckdb
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
import jqdatasdk as jq
import tushare as ts

In [14]:
from pathlib import Path

print(Path("."))
print(Path.cwd())
print(Path(".").resolve())

.
E:\Latitude_Analytics_v2\00_draft_collection_01
E:\Latitude_Analytics_v2\00_draft_collection_01


In [2]:
project_markers = [".git", ".env", "config/settings.py"]
current_path = Path.cwd().resolve()

for candidate_root in [current_path, *current_path.parents]:
    if all((candidate_root / marker).exists() for marker in project_markers):
        sys.path.insert(0, str(candidate_root))
        break
else:
    raise RuntimeError("未找到项目根目录")

from config.settings import settings

In [4]:
Path.cwd() # current working directory

WindowsPath('E:/Latitude_Analytics_v2/00_draft_collection_01')

In [5]:
Path.cwd().resolve()

WindowsPath('E:/Latitude_Analytics_v2/00_draft_collection_01')

In [7]:
pathlib.WindowsPath('E:/Latitude_Analytics_v2/R00_draft_collection_01')

WindowsPath('E:/Latitude_Analytics_v2/00_draft_collection_01')

In [3]:
pro = ts.pro_api(settings.tushare_token)
jq.auth(settings.jqdata_id, settings.jqdata_secret)

auth success 


In [ ]:
def write_dataset(df, path, partitions):
    path.mkdir(parents=True, exist_ok=True)
    ds.write_dataset(
        pa.Table.from_pandas(df, preserve_index=False),
        base_dir=str(path),
        format="parquet",
        partitioning=ds.partitioning(
            pa.schema([(c, pa.string()) for c in partitions]),
            flavor="hive",
        ),
        existing_data_behavior="delete_matching",
    )

def normalize_contracts(raw):
    df = raw.rename_axis("jq_code").reset_index()
    df["symbol"] = df["jq_code"].str.split(".").str[0]
    df["exchange"] = df["jq_code"].str.split(".").str[1]
    df["variety"] = df["symbol"].str.extract(r"^([A-Za-z]+)", expand=False).str.upper()
    df["exchange_cn"] = df["exchange"].map({
        "XSGE": "上期所", "XDCE": "大商所", "XZCE": "郑商所",
        "CCFX": "中金所", "XINE": "能源中心", "XGFE": "广期所", "GFEX": "广期所",
    }).fillna(df["exchange"])
    df["is_active"] = pd.to_datetime(df["end_date"]) >= pd.Timestamp.today().normalize()
    df["source"] = "JQData"
    df["updated_at"] = pd.Timestamp.now()
    return df[[
        "jq_code", "symbol", "variety", "exchange", "exchange_cn",
        "display_name", "name", "start_date", "end_date", "type",
        "is_active", "source", "updated_at"
    ]]

def normalize_daily(raw):
    df = raw.reset_index().rename(columns={"time": "trade_date", "code": "jq_code"})
    df["trade_date"] = pd.to_datetime(df["trade_date"]).dt.date
    df["symbol"] = df["jq_code"].str.split(".").str[0]
    df["exchange"] = df["jq_code"].str.split(".").str[1]
    df["variety"] = df["symbol"].str.extract(r"^([A-Za-z]+)", expand=False).str.upper()
    df["source"] = "JQData"
    df["updated_at"] = pd.Timestamp.now()
    df["year"] = pd.to_datetime(df["trade_date"]).dt.year.astype(str)
    return df

# 1. 拉期货合约维表
contracts = normalize_contracts(jq.get_all_securities(types=["futures"]))
write_dataset(
    contracts,
    LAKE_ROOT / "silver" / "dim_futures_contract",
    ["source"],
)

# 2. 选两个仍活跃的合约，拉日线
symbols = contracts.query("is_active").head(2)["jq_code"].tolist()
end_date = date.today()
start_date = end_date - timedelta(days=20)

daily = jq.get_price(
    symbols,
    start_date=start_date.isoformat(),
    end_date=end_date.isoformat(),
    frequency="daily",
    fields=["open", "high", "low", "close", "volume", "money", "open_interest"],
    panel=False,
)
daily = normalize_daily(daily)

write_dataset(
    daily,
    LAKE_ROOT / "silver" / "fact_futures_daily",
    ["source", "exchange", "variety", "year"],
)

# 3. DuckDB 只建 view，不复制大表
con = duckdb.connect(str(LAKE_ROOT / "jqdata.duckdb"))

con.execute(f"""
CREATE OR REPLACE VIEW dim_futures_contract AS
SELECT * FROM read_parquet(
  '{(LAKE_ROOT / "silver" / "dim_futures_contract" / "**" / "*.parquet").as_posix()}',
  hive_partitioning = 1
)
""")

con.execute(f"""
CREATE OR REPLACE VIEW fact_futures_daily AS
SELECT * FROM read_parquet(
  '{(LAKE_ROOT / "silver" / "fact_futures_daily" / "**" / "*.parquet").as_posix()}',
  hive_partitioning = 1
)
""")

print(con.execute("""
SELECT variety, min(trade_date), max(trade_date), count(*) AS rows
FROM fact_futures_daily
GROUP BY variety
ORDER BY variety
""").df())